In [2]:
# import packages
import pandas as pd
import os

In [3]:
# input file(s) - '*_merged_rank{}_aff{}.csv'
file_paths = ["/private10/Projects/Efi/CRG/SF3B1_WT/SplicingAnalysis/_forNEanalysis/UM_X/SplicingEvents/DMSO_Indisulam_merged_rank0.5_aff50.0.csv", 
"/private10/Projects/Efi/CRG/SF3B1_WT/SplicingAnalysis/_forNEanalysis/UM_X/SplicingEvents/DMSO_PladienolideB_merged_rank0.5_aff50.0.csv"]
# output directory
out_dir = '/private10/Projects/Efi/CRG/SF3B1_WT/SplicingAnalysis/_forNEanalysis/UM_X/SplicingEvents/'

In [4]:
# merge all files into one df and remove duplicates
dataframes = [pd.read_csv(file) for file in file_paths]
combined_dataframe = pd.concat(dataframes, ignore_index=True)
combined_dataframe_noDups = combined_dataframe.drop_duplicates()

In [5]:
# melt the data according to rank and nM
id_cols = ['Group','Splicing Event','Peptide','ID','SplicingIndex', 'Avg.PSI_PeptideSource','Avg.PSI_OtherGroup','Avg.TPM_PeptideSource','Avg.TPM_OtherGroup'] 
data = combined_dataframe_noDups
df_rank = data.melt(id_vars=id_cols, 
                value_vars= data.columns[data.columns.str.endswith('_Rank')],
                var_name='HLA_Rank', 
                value_name='Rank').dropna(subset=['Rank'])
df_nM = data.melt(id_vars=id_cols, 
                value_vars= data.columns[data.columns.str.endswith('_nM')],
                var_name='HLA_nM', 
                value_name='nM').dropna(subset=['nM'])
df_rank['HLA'] = df_rank['HLA_Rank'].str.split('_').str[0]
df_nM['HLA'] = df_nM['HLA_nM'].str.split('_').str[0] 

df_rank.drop(columns=['HLA_Rank'], inplace=True)
df_nM.drop(columns=['HLA_nM'], inplace=True)

# Rearrange columns in each melted DataFrame
df_rank = df_rank[id_cols+['HLA', 'Rank']]
df_nM = df_nM[id_cols+['HLA', 'nM']]

df_merged = pd.merge(df_rank, df_nM, on=id_cols+['HLA'], how='inner')

In [6]:
# Identify peptides in both control and treatments
#control_peptides = set(df_merged[(df_merged['Group'] == 'Mock6') | (df_merged['Group'] == 'NoTreatmentSF')]['Peptide'])
#treatment_peptides = set(df_merged[(df_merged['Group'] == 'Indisulam') | (df_merged['Group'] == 'PladB')]['Peptide'])
control_group_names = ['DMSO']
control_peptides = set(df_merged[df_merged['Group'].isin(control_group_names)]['Peptide'])
treatment_group_names = ['Indisulam', 'PladienolideB']
treatment_peptides = set(df_merged[df_merged['Group'].isin(treatment_group_names)]['Peptide'])
# Peptides in both control and treatments
common_peptides = control_peptides.intersection(treatment_peptides)

# Filter the data frame
filtered_df = df_merged[~(df_merged['Peptide'].isin(common_peptides))]

# Identify peptides in both control and treatments (for another comparisons of control vs. treatments)
#control_peptides = set(df_merged[(df_merged['Group'] == 'Mock18') | (df_merged['Group'] == 'NoTreatmentSF')]['Peptide'])
#treatment_peptides = set(df_merged[(df_merged['Group'] == '5Aza') | (df_merged['Group'] == 'FB23-2')]['Peptide'])
#control_group_names = ['Mock18', 'NoTreatmentSF']
#control_peptides = set(df_merged[df_merged['Group'].isin(control_group_names)]['Peptide'])
#treatment_group_names = ['5Aza', 'FB23-2']
#treatment_peptides = set(df_merged[df_merged['Group'].isin(treatment_group_names)]['Peptide'])
# Peptides in both control and treatments
#common_peptides = control_peptides.intersection(treatment_peptides)
# Filter the data frame
#filtered_df = filtered_df[~(filtered_df['Peptide'].isin(common_peptides))]

In [7]:
# save the file
out_file = os.path.join(out_dir, 'NovelStrongBindingEpitopes_noDups.csv')
filtered_df.to_csv(out_file, index=False)